In [17]:
import pandas as pd
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
import gradio as gr



In [18]:

def load_csv_safely(filepath, description=""):
    """安全載入CSV檔案，包含錯誤處理"""
    try:
        if not os.path.exists(filepath):
            print(f"警告: {filepath} 檔案不存在")
            return None
        
        df = pd.read_csv(filepath)
        print(f"✓ 成功載入 {description}: {df.shape[0]} 列, {df.shape[1]} 欄")
        return df
    except Exception as e:
        print(f"✗ 載入 {filepath} 時發生錯誤: {e}")
        return None

def standardize_column_names(df, mappings, df_name=""):
    """標準化欄位名稱"""
    original_columns = df.columns.tolist()
    
    for old_name, new_name in mappings.items():
        if old_name in df.columns and new_name not in df.columns:
            df = df.rename(columns={old_name: new_name})
            print(f"  {df_name}: 重新命名 '{old_name}' -> '{new_name}'")
    
    return df

def validate_merge_keys(df1, df2, keys, df1_name="", df2_name=""):
    """驗證合併鍵的有效性"""
    missing_keys_df1 = [key for key in keys if key not in df1.columns]
    missing_keys_df2 = [key for key in keys if key not in df2.columns]
    
    if missing_keys_df1:
        print(f"警告: {df1_name} 缺少合併鍵: {missing_keys_df1}")
    if missing_keys_df2:
        print(f"警告: {df2_name} 缺少合併鍵: {missing_keys_df2}")
    
    return len(missing_keys_df1) == 0 and len(missing_keys_df2) == 0

def merge_with_info(df1, df2, keys, how='left', df1_name="", df2_name="", suffixes=('', '_right')):
    """執行合併並顯示資訊"""
    if not validate_merge_keys(df1, df2, keys, df1_name, df2_name):
        print(f"✗ 無法合併 {df1_name} 和 {df2_name}: 缺少必要的合併鍵")
        return df1
    
    original_rows = len(df1)
    merged_df = pd.merge(df1, df2, how=how, on=keys, suffixes=suffixes)
    new_rows = len(merged_df)
    
    print(f"✓ 合併 {df1_name} + {df2_name}: {original_rows} -> {new_rows} 列")
    
    # 檢查是否有新的欄位被加入
    new_columns = set(merged_df.columns) - set(df1.columns)
    if new_columns:
        print(f"  新增欄位: {list(new_columns)}")
    
    return merged_df

def main():
    print("=== F1 數據整合工具 ===\n")
    
    # 1. 載入所有CSV檔案
    print("1. 載入數據檔案...")
    file_configs = {
        'drivers': ('./data/drivers_updated.csv', '車手數據'),
        'laps': ('./data/fastest_laps_updated.csv', '最快圈速數據'),
        'teams': ('./data/teams_updated.csv', '車隊數據'),
        'winners': ('./data/winners.csv', '比賽勝者數據')
    }
    
    data = {}
    for key, (filepath, description) in file_configs.items():
        df = load_csv_safely(filepath, description)
        if df is not None:
            data[key] = df
        else:
            print(f"跳過 {key} 數據...")
    
    if not data:
        print("✗ 沒有成功載入任何數據檔案")
        return
    
    print(f"\n成功載入 {len(data)} 個數據檔案\n")
    
    # 2. 標準化欄位名稱
    print("2. 標準化欄位名稱...")
    column_mappings = {
        'Car': 'Team',
        'Winner': 'Driver'
    }
    
    for key, df in data.items():
        data[key] = standardize_column_names(df, column_mappings, key)
        print(f"  {key} 欄位: {list(data[key].columns)}")
    
    print()
    
    # 3. 開始合併數據
    print("3. 合併數據...")
    
    # 從drivers開始，如果沒有drivers就用其他數據作為起點
    if 'drivers' in data:
        result_df = data['drivers'].copy()
        current_name = 'drivers'
    else:
        # 找第一個可用的數據集
        first_key = list(data.keys())[0]
        result_df = data[first_key].copy()
        current_name = first_key
        print(f"使用 {first_key} 作為起始數據集")
    
    # 合併 laps 數據
    if 'laps' in data and 'drivers' in data:
        merge_keys = ['Driver', 'Team', 'year']
        result_df = merge_with_info(
            result_df, data['laps'], merge_keys, 
            df1_name=current_name, df2_name='laps'
        )
        current_name += '+laps'
    
    # 合併 teams 數據
    if 'teams' in data:
        merge_keys = ['Team', 'year']
        result_df = merge_with_info(
            result_df, data['teams'], merge_keys,
            df1_name=current_name, df2_name='teams',
            suffixes=('', '_team')
        )
        current_name += '+teams'
    
    # 合併 winners 數據（如果有Grand Prix欄位）
    if 'winners' in data:
        if 'Grand Prix' in data['winners'].columns:
            merge_keys = ['Grand Prix', 'Driver', 'Team', 'year']
            result_df = merge_with_info(
                result_df, data['winners'], merge_keys,
                df1_name=current_name, df2_name='winners',
                suffixes=('', '_winner')
            )
            current_name += '+winners'
        else:
            print("  winners 數據缺少 'Grand Prix' 欄位，跳過合併")
    
    # 4. 數據品質檢查
    print(f"\n4. 數據品質檢查...")
    print(f"最終數據集形狀: {result_df.shape}")
    print(f"總欄位數: {len(result_df.columns)}")
    
    # 檢查缺失值
    missing_summary = result_df.isnull().sum()
    missing_cols = missing_summary[missing_summary > 0]
    if len(missing_cols) > 0:
        print("缺失值統計:")
        for col, count in missing_cols.items():
            percentage = (count / len(result_df)) * 100
            print(f"  {col}: {count} ({percentage:.1f}%)")
    else:
        print("✓ 沒有發現缺失值")
    
    # 檢查重複列
    duplicates = result_df.duplicated().sum()
    if duplicates > 0:
        print(f"發現 {duplicates} 個重複列")
    else:
        print("✓ 沒有發現重複列")
    
    # 5. 輸出結果
    print(f"\n5. 輸出結果...")
    output_path = './data/f1_merged_all.csv'
    
    try:
        result_df.to_csv(output_path, index=False, encoding='utf-8')
        print(f"✓ 成功輸出到 {output_path}")
        print(f"檔案大小: {os.path.getsize(output_path) / 1024:.1f} KB")
    except Exception as e:
        print(f"✗ 輸出檔案時發生錯誤: {e}")
    
    # 顯示前幾列作為預覽
    print(f"\n數據預覽 (前5列):")
    print(result_df.head())
    
    return result_df

if __name__ == "__main__":
    merged_data = main()

=== F1 數據整合工具 ===

1. 載入數據檔案...
✓ 成功載入 車手數據: 1661 列, 7 欄
✓ 成功載入 最快圈速數據: 1108 列, 6 欄
✓ 成功載入 車隊數據: 695 列, 4 欄
✓ 成功載入 比賽勝者數據: 1110 列, 7 欄

成功載入 4 個數據檔案

2. 標準化欄位名稱...
  drivers: 重新命名 'Car' -> 'Team'
  drivers 欄位: ['Pos', 'Driver', 'Nationality', 'Team', 'PTS', 'year', 'Code']
  laps: 重新命名 'Car' -> 'Team'
  laps 欄位: ['Grand Prix', 'Driver', 'Team', 'Time', 'year', 'Code']
  teams 欄位: ['Pos', 'Team', 'PTS', 'year']
  winners: 重新命名 'Car' -> 'Team'
  winners: 重新命名 'Winner' -> 'Driver'
  winners 欄位: ['Grand Prix', 'Date', 'Driver', 'Team', 'Laps', 'Time', 'Name Code']

3. 合併數據...
✓ 合併 drivers + laps: 1661 -> 2284 列
  新增欄位: ['Code_right', 'Grand Prix', 'Time']
✓ 合併 drivers+laps + teams: 2284 -> 2284 列
  新增欄位: ['Pos_team', 'PTS_team']
警告: winners 缺少合併鍵: ['year']
✗ 無法合併 drivers+laps+teams 和 winners: 缺少必要的合併鍵

4. 數據品質檢查...
最終數據集形狀: (2284, 12)
總欄位數: 12
缺失值統計:
  Team: 11 (0.5%)
  Grand Prix: 1224 (53.6%)
  Time: 1225 (53.6%)
  Code_right: 1224 (53.6%)
  Pos_team: 264 (11.6%)
  PTS_team: 264 (11.6%)


In [ ]:


class F1PredictionSystem:
    def __init__(self):
        self.driver_skill_model = None
        self.team_skill_model = None
        self.ranking_model = None
        self.scalers = {}
        self.encoders = {}
        self.driver_names = []
        self.team_names = []
        self.circuit_names = []
        self.active_drivers = {}  # 活躍車手及其當前車隊
        self.driver_team_history = {}  # 車手-車隊歷史記錄
        self.training_data = None
        self.all_drivers_data = {}  # 存儲所有車手的完整數據
        
    def prepare_data(self, df):
        print("準備數據中...")
        self.training_data = df.copy()
        
        df_clean = df.copy()
        
        # Define a placeholder for missing values
        nan_placeholder = "Unknown_Value"

        # Clean data and fill NaNs consistently
        df_clean['Driver'] = df_clean['Driver'].str.strip().fillna(nan_placeholder)
        df_clean['Team'] = df_clean['Team'].str.strip().fillna(nan_placeholder)
        
        if 'Grand Prix' in df_clean.columns:
            df_clean['Grand Prix'] = df_clean['Grand Prix'].str.strip().fillna(nan_placeholder)

        # Store complete driver data for dynamic filtering
        self._store_complete_driver_data(df_clean)
        
        # Identify active drivers (now uses data with NaNs filled)
        self._identify_active_drivers(df_clean) 
        
        # Establish encoders
        self.driver_names = sorted(df_clean['Driver'].unique().tolist())
        self.team_names = sorted(df_clean['Team'].unique().tolist())
        
        if 'Grand Prix' in df_clean.columns:
            self.circuit_names = sorted(df_clean['Grand Prix'].unique().tolist())
        else:
            self.circuit_names = ['Monaco', 'Silverstone', 'Monza']

        self.encoders['driver'] = LabelEncoder().fit(self.driver_names)
        self.encoders['team'] = LabelEncoder().fit(self.team_names)
        self.encoders['circuit'] = LabelEncoder().fit(self.circuit_names)
        
        # Process data for encoding
        df_processed = df_clean.copy() 
        
        df_processed['driver_encoded'] = self.encoders['driver'].transform(df_processed['Driver'])
        df_processed['team_encoded'] = self.encoders['team'].transform(df_processed['Team'])
        
        if 'Grand Prix' in df_processed.columns:
            df_processed['circuit_encoded'] = self.encoders['circuit'].transform(df_processed['Grand Prix'])
        else:
            if len(self.encoders['circuit'].classes_) > 0:
                df_processed['circuit_encoded'] = np.random.choice(
                    len(self.encoders['circuit'].classes_),
                    size=len(df_processed)
                )
            else:
                df_processed['circuit_encoded'] = 0 
                print("Warning: Circuit encoder has no classes. Defaulting circuit_encoded to 0.")

        df_processed = self.create_skill_features(df_processed)
        return df_processed

    def _store_complete_driver_data(self, df):
        """存儲所有車手的完整數據，用於動態篩選"""
        print("存儲車手完整數據...")
        
        # 如果沒有年份欄位，使用數據順序
        if 'year' not in df.columns:
            df = df.copy()
            df['year'] = range(len(df))
        
        # 為每個車手存儲完整的歷史數據
        for driver in df['Driver'].unique():
            driver_data = df[df['Driver'] == driver].copy()
            driver_data = driver_data.sort_values('year')
            
            # 計算車手的活躍年份範圍
            min_year = driver_data['year'].min()
            max_year = driver_data['year'].max()
            total_races = len(driver_data)
            
            # 獲取每年的車隊信息
            yearly_teams = driver_data.groupby('year')['Team'].first().to_dict()
            
            # 計算與各車隊的合作次數
            team_history = driver_data.groupby('Team').size().sort_values(ascending=False).to_dict()
            
            self.all_drivers_data[driver] = {
                'data': driver_data,
                'min_year': min_year,
                'max_year': max_year,
                'total_races': total_races,
                'yearly_teams': yearly_teams,
                'team_history': team_history
            }
        
        print(f"存儲了 {len(self.all_drivers_data)} 位車手的完整數據")

    def get_active_drivers_for_year(self, target_year, activity_window=5, min_drivers=15):
        """根據指定年份獲取活躍車手列表"""
        print(f"獲取 {target_year} 年的活躍車手...")
        
        active_drivers = {}
        
        # 為每個車手計算在目標年份的活躍程度
        for driver, data_info in self.all_drivers_data.items():
            min_year = data_info['min_year']
            max_year = data_info['max_year']
            yearly_teams = data_info['yearly_teams']
            
            # 判斷車手是否在目標年份附近活躍
            years_since_last_race = target_year - max_year
            years_to_first_race = min_year - target_year
            
            # 活躍判斷條件：
            # 1. 在目標年份之前活動窗口內有比賽記錄
            # 2. 或者在目標年份之後不久開始職業生涯
            is_active = (years_since_last_race <= activity_window and years_since_last_race >= 0) or \
                       (years_to_first_race <= 2 and years_to_first_race >= 0) or \
                       (min_year <= target_year <= max_year)
            
            if is_active:
                # 確定該車手在目標年份的車隊
                current_team = self._get_team_for_year(driver, target_year, yearly_teams)
                
                # 計算活躍度分數（越近期越高分）
                if min_year <= target_year <= max_year:
                    activity_score = 100  # 正在活躍期
                elif years_since_last_race >= 0:
                    activity_score = max(0, 80 - years_since_last_race * 10)  # 近期退役
                else:
                    activity_score = max(0, 60 - years_to_first_race * 15)  # 即將出道
                
                active_drivers[driver] = {
                    'current_team': current_team,
                    'target_year': target_year,
                    'last_active_year': max_year,
                    'first_active_year': min_year,
                    'total_races': data_info['total_races'],
                    'activity_score': activity_score,
                    'years_since_last_race': years_since_last_race
                }
        
        # 如果活躍車手太少，放寬條件
        if len(active_drivers) < min_drivers:
            print(f"活躍車手僅 {len(active_drivers)} 位，放寬條件...")
            
            # 按活躍度分數排序所有車手
            all_candidates = []
            for driver, data_info in self.all_drivers_data.items():
                min_year = data_info['min_year']
                max_year = data_info['max_year']
                years_since_last_race = target_year - max_year
                years_to_first_race = min_year - target_year
                
                # 計算擴展的活躍度分數
                if min_year <= target_year <= max_year:
                    score = 100
                elif years_since_last_race >= 0:
                    score = max(0, 90 - years_since_last_race * 5)
                else:
                    score = max(0, 70 - years_to_first_race * 8)
                
                current_team = self._get_team_for_year(driver, target_year, data_info['yearly_teams'])
                
                all_candidates.append({
                    'driver': driver,
                    'score': score,
                    'current_team': current_team,
                    'total_races': data_info['total_races'],
                    'last_active_year': max_year,
                    'first_active_year': min_year,
                    'years_since_last_race': years_since_last_race
                })
            
            # 按分數排序並選取前面的車手
            all_candidates.sort(key=lambda x: (x['score'], x['total_races']), reverse=True)
            
            active_drivers = {}
            for candidate in all_candidates[:max(min_drivers, len(active_drivers))]:
                driver = candidate['driver']
                active_drivers[driver] = {
                    'current_team': candidate['current_team'],
                    'target_year': target_year,
                    'last_active_year': candidate['last_active_year'],
                    'first_active_year': candidate['first_active_year'],
                    'total_races': candidate['total_races'],
                    'activity_score': candidate['score'],
                    'years_since_last_race': candidate['years_since_last_race']
                }
        
        print(f"為 {target_year} 年選定 {len(active_drivers)} 位活躍車手")
        return active_drivers

    def _get_team_for_year(self, driver, target_year, yearly_teams):
        """獲取車手在指定年份的車隊"""
        # 如果該年份有確切記錄
        if target_year in yearly_teams:
            return yearly_teams[target_year]
        
        # 找最接近的年份
        available_years = list(yearly_teams.keys())
        if not available_years:
            return "Unknown Team"
        
        # 找最接近但不超過目標年份的年份
        past_years = [y for y in available_years if y <= target_year]
        if past_years:
            closest_year = max(past_years)
            return yearly_teams[closest_year]
        
        # 如果沒有過去的記錄，使用最早的記錄
        earliest_year = min(available_years)
        return yearly_teams[earliest_year]

    def _identify_active_drivers(self, df):
        """識別活躍車手並確定他們的當前車隊"""
        print("識別活躍車手...")
        
        # 如果沒有年份欄位，使用數據順序
        if 'year' not in df.columns:
            df = df.copy()
            df['year'] = range(len(df))
        
        # 使用當前最大年份作為基準
        current_max_year = df['year'].max()
        self.active_drivers = self.get_active_drivers_for_year(current_max_year)
        
        # 建立車手歷史記錄
        for driver in df['Driver'].unique():
            driver_data = df[df['Driver'] == driver]
            team_history = driver_data.groupby('Team').size().sort_values(ascending=False)
            self.driver_team_history[driver] = team_history.to_dict()
        
        print(f"初始識別出 {len(self.active_drivers)} 位活躍車手")

    def update_active_drivers_for_year(self, year):
        """更新指定年份的活躍車手"""
        self.active_drivers = self.get_active_drivers_for_year(int(year))
        active_driver_list = list(self.active_drivers.keys())
        active_driver_list.sort()
        return active_driver_list

    def create_skill_features(self, df):
        """創建車手和車隊實力特徵"""
        print("計算實力特徵...")
        
        df = df.sort_values(['year', 'Driver']).reset_index(drop=True)
        df['driver_skill'] = 0.0
        df['team_skill'] = 0.0
        
        # 檢查位置欄位
        pos_col = None
        if 'Pos' in df.columns:
            pos_col = 'Pos'
            print(f"使用位置欄位: {pos_col}")
        elif 'Position' in df.columns:
            pos_col = 'Position'
            print(f"使用位置欄位: {pos_col}")
        else:
            print("警告：找不到位置欄位，將使用隨機數據")
        
        # 改進的成績評分系統
        def calculate_position_score(pos):
            if pd.isna(pos):
                return 0  # 退賽或DNS給0分
            
            # 處理各種可能的位置格式
            try:
                if isinstance(pos, str):
                    # 移除可能的特殊字符，只保留數字
                    pos_clean = ''.join(filter(str.isdigit, pos))
                    if not pos_clean:
                        return 0  # 如果沒有數字，視為無效
                    pos = float(pos_clean)
                else:
                    pos = float(pos)
                    
                if pos <= 0:
                    return 0
                    
            except (ValueError, TypeError):
                return 0
            
            # 嚴格的評分標準
            if pos == 1:
                return 100
            elif pos == 2:
                return 85
            elif pos == 3:
                return 70
            elif pos == 4:
                return 60
            elif pos == 5:
                return 50
            elif pos <= 10:
                return 45 - (pos-6) * 5  # 6-10名：40,35,30,25,20
            elif pos <= 20:
                return 15 - (pos-11) * 1  # 11-20名：14到5
            else:
                return max(3 - (pos-21) * 0.2, 0)
        
        # 計算位置分數
        if pos_col:
            df['position_score'] = df[pos_col].apply(calculate_position_score)
        else:
            # 如果沒有位置數據，使用保守的隨機分數
            np.random.seed(42)  # 確保可重現
            df['position_score'] = np.random.exponential(scale=15, size=len(df))
            df['position_score'] = np.clip(df['position_score'], 0, 50)
        
        
        # 重新設計車手實力計算
        driver_skills = {}
        
        for driver in df['Driver'].unique():
            driver_data = df[df['Driver'] == driver].sort_values('year')
            valid_scores = driver_data['position_score'].dropna()
            
            if len(valid_scores) == 0:
                skill = 5  # 沒有有效數據給極低分
            elif len(valid_scores) < 5:
                # 數據太少，給予懲罰
                skill = valid_scores.mean() * 0.5
            else:
                # 使用加權平均，近期表現權重更高
                weights = np.exp(np.linspace(-1, 0, len(valid_scores)))
                skill = np.average(valid_scores, weights=weights)
                
                # 根據一致性調整 - 表現不穩定的車手降分
                consistency_penalty = valid_scores.std() / 20
                skill = skill - consistency_penalty
            
            driver_skills[driver] = max(skill, 0)
        
        
        # 將技能分數分配到dataframe
        for driver, skill in driver_skills.items():
            df.loc[df['Driver'] == driver, 'driver_skill'] = skill
        
        # 車隊實力計算（簡化版本）
        for team in df['Team'].unique():
            team_data = df[df['Team'] == team]
            # 使用該車隊車手的平均實力作為車隊實力
            team_skill = team_data['driver_skill'].mean()
            df.loc[df['Team'] == team, 'team_skill'] = team_skill
        
        # 標準化處理
        print("\n=== 標準化前後對照 ===")
        before_top5 = df.groupby('Driver')['driver_skill'].mean().sort_values(ascending=False).head(5)
        print("標準化前前5名:")
        for driver, skill in before_top5.items():
            print(f"  {driver}: {skill:.2f}")
        
        # 使用RobustScaler而不是MinMaxScaler，更能處理異常值
        from sklearn.preprocessing import RobustScaler
        
        robust_scaler = RobustScaler()
        df['driver_skill'] = robust_scaler.fit_transform(df[['driver_skill']]).flatten()
        df['team_skill'] = robust_scaler.fit_transform(df[['team_skill']]).flatten()
        
        # 重新映射到0-100範圍
        from sklearn.preprocessing import MinMaxScaler
        final_scaler = MinMaxScaler(feature_range=(0, 100))
        df['driver_skill'] = final_scaler.fit_transform(df[['driver_skill']]).flatten()
        df['team_skill'] = final_scaler.fit_transform(df[['team_skill']]).flatten()
        
        after_top5 = df.groupby('Driver')['driver_skill'].mean().sort_values(ascending=False).head(5)
        print("標準化後前5名:")
        for driver, skill in after_top5.items():
            print(f"  {driver}: {skill:.2f}")
        
        return df

    def build_skill_prediction_model(self, input_dim):
        """建立實力預測模型"""
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(input_dim,)),
            tf.keras.layers.Dense(128, activation='relu'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Dense(32, activation='relu'),
            tf.keras.layers.Dense(2, activation='sigmoid')
        ])
        
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
            loss='huber',
            metrics=['mae']
        )
        return model

    def build_ranking_model(self, input_dim):
        """建立排名預測模型"""
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(input_dim,)),
            tf.keras.layers.Dense(256, activation='relu'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.4),
            tf.keras.layers.Dense(128, activation='relu'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Dense(32, activation='relu'),
            tf.keras.layers.Dense(1, activation='linear')
        ])
        
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
            loss='huber',
            metrics=['mae']
        )
        return model

    def build_lstm_skill_model(self, sequence_length, feature_dim):
        """建立LSTM實力預測模型"""
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(sequence_length, feature_dim)),
            tf.keras.layers.LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.2),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.LSTM(64, return_sequences=False, dropout=0.2, recurrent_dropout=0.1),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dense(32, activation='relu'),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Dense(2, activation='sigmoid')
        ])
        
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
            loss='huber',
            metrics=['mae']
        )
        return model

    def build_lstm_ranking_model(self, sequence_length, feature_dim):
        """建立LSTM排名預測模型"""
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(sequence_length, feature_dim)),
            tf.keras.layers.LSTM(256, return_sequences=True, dropout=0.4, recurrent_dropout=0.3),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.2),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.LSTM(64, return_sequences=False, dropout=0.2, recurrent_dropout=0.1),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dense(32, activation='relu'),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Dense(1, activation='linear')
        ])
        
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
            loss='huber',
            metrics=['mae']
        )
        return model

    def prepare_sequence_data(self, df, sequence_length=5):
        """準備LSTM所需的序列數據"""
        print(f"準備序列數據，序列長度: {sequence_length}")
        df_processed = self.prepare_data(df)
        
        # 按車手和時間排序
        df_processed = df_processed.sort_values(['driver_encoded', 'year']).reset_index(drop=True)
        
        sequences = []
        targets_skill = []
        targets_ranking = []
        
        # 為每個車手創建序列
        for driver_encoded in df_processed['driver_encoded'].unique():
            driver_data = df_processed[df_processed['driver_encoded'] == driver_encoded].copy()
            
            if len(driver_data) < sequence_length + 1:
                continue
                
            for i in range(len(driver_data) - sequence_length):
                # 輸入序列（前sequence_length個時間點）
                sequence_data = driver_data.iloc[i:i+sequence_length]
                
                # 目標值（下一個時間點）
                target_data = driver_data.iloc[i+sequence_length]
                
                # 準備序列特徵
                sequence_features = []
                for _, row in sequence_data.iterrows():
                    features = [
                        row['driver_encoded'],
                        row['team_encoded'],
                        row['circuit_encoded'] if 'circuit_encoded' in row else 0,
                        row['year'],
                        row['driver_skill'],
                        row['team_skill']
                    ]
                    sequence_features.append(features)
                
                sequences.append(sequence_features)
                targets_skill.append([target_data['driver_skill']/100.0, target_data['team_skill']/100.0])
                
                # 排名目標
                if 'Pos' in target_data:
                    ranking_target = pd.to_numeric(target_data['Pos'], errors='coerce')
                    if pd.isna(ranking_target):
                        ranking_target = 12
                elif 'Position' in target_data:
                    ranking_target = pd.to_numeric(target_data['Position'], errors='coerce')
                    if pd.isna(ranking_target):
                        ranking_target = 12
                else:
                    ranking_target = 12
                    
                targets_ranking.append(ranking_target)
        
        return np.array(sequences), np.array(targets_skill), np.array(targets_ranking)

    def train_models(self, df, sequence_length=5):
        """訓練LSTM預測模型"""
        print("開始訓練LSTM模型...")
        
        # 準備序列數據
        X_sequences, y_skill_seq, y_ranking_seq = self.prepare_sequence_data(df, sequence_length)
        

        
        if len(X_sequences) == 0:
            raise ValueError("序列數據不足，無法訓練LSTM模型")
        
        print(f"準備了 {len(X_sequences)} 個序列")
        
        # 標準化序列數據
        X_sequences_reshaped = X_sequences.reshape(-1, X_sequences.shape[-1])
        self.scalers['lstm_sequence'] = StandardScaler()
        X_sequences_scaled = self.scalers['lstm_sequence'].fit_transform(X_sequences_reshaped)
        X_sequences_scaled = X_sequences_scaled.reshape(X_sequences.shape)
        
        # 分割訓練和測試數據
        split_idx = int(0.8 * len(X_sequences_scaled))
        X_seq_train, X_seq_test = X_sequences_scaled[:split_idx], X_sequences_scaled[split_idx:]
        y_skill_train, y_skill_test = y_skill_seq[:split_idx], y_skill_seq[split_idx:]
        y_rank_train, y_rank_test = y_ranking_seq[:split_idx], y_ranking_seq[split_idx:]
        
        # 訓練LSTM實力預測模型
        print("訓練LSTM參數預測模型...")
        self.lstm_skill_model = self.build_lstm_skill_model(sequence_length, X_sequences.shape[-1])
        
        early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=15, restore_best_weights=True)
        
        reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=8, min_lr=1e-6)
        
        self.lstm_skill_model.fit(
            X_seq_train, y_skill_train,
            validation_data=(X_seq_test, y_skill_test),
            epochs=100,
            batch_size=32,
            callbacks=[early_stopping, reduce_lr],
            verbose=1
        )
        
        # 訓練LSTM排名預測模型
        print("訓練LSTM排名預測模型...")
        self.lstm_ranking_model = self.build_lstm_ranking_model(sequence_length, X_sequences.shape[-1])
        
        self.lstm_ranking_model.fit(
            X_seq_train, y_rank_train,
            validation_data=(X_seq_test, y_rank_test),
            epochs=100,
            batch_size=32,
            callbacks=[early_stopping, reduce_lr],
            verbose=1
        )
        
        self.df = df.copy()
        #print(self.df.columns)


        self.sequence_length = sequence_length
        self.use_lstm = True
        
        print("LSTM模型訓練完成！")

    def get_driver_history_sequence(self, driver_name, circuit_name, year, sequence_length=None, df_processed=None):
        """獲取車手的歷史序列數據用於LSTM預測（多層級 fallback，並保證資料一致性）"""
        if sequence_length is None:
            sequence_length = getattr(self, 'sequence_length', 5)
        if driver_name not in self.active_drivers:
            return None

        current_team = self.active_drivers[driver_name]['current_team']
        try:
            # === 重點：用和訓練同樣的前處理 ===
            
            
            driver_encoded = self.encoders['driver'].transform([driver_name])[0]
            team_encoded = self.encoders['team'].transform([current_team])[0]
            circuit_encoded = self.encoders['circuit'].transform([circuit_name])[0]

            sequence = []
            for i in range(sequence_length):
                historical_year = year - sequence_length + i + 1

                # 1. 精確查詢 driver+team+circuit+year
                row = df_processed[
                    (df_processed['Driver'] == driver_name) &
                    (df_processed['Team'] == current_team) &
                    (df_processed['Grand Prix'] == circuit_name) &
                    (df_processed['year'] == historical_year)
                ]
                # 2. 只查 driver+year
                if row.empty:
                    row = df_processed[
                        (df_processed['Driver'] == driver_name) &
                        (df_processed['year'] == historical_year)
                    ]
                # 3. 只查 team+year
                if row.empty:
                    row = df_processed[
                        (df_processed['Team'] == current_team) &
                        (df_processed['year'] == historical_year)
                    ]
                # 4. 再查不到就用全資料均值
                if not row.empty:
                    driver_skill = row['driver_skill'].mean()
                    team_skill = row['team_skill'].mean()
                else:
                    driver_skill = df_processed['driver_skill'].mean()
                    team_skill = df_processed['team_skill'].mean()

                sequence.append([
                    driver_encoded,
                    team_encoded,
                    circuit_encoded,
                    historical_year,
                    driver_skill,
                    team_skill
                ])

            return np.array([sequence])
        except Exception as e:
            print(f"獲取 {driver_name} 歷史序列時發生錯誤: {e}")
            return None




    def predict_driver_performance(self, driver_name, circuit_name, year, df_processed=None):
        """預測單個車手的表現（使用LSTM）"""
        if driver_name not in self.active_drivers:
            return None
            
        current_team = self.active_drivers[driver_name]['current_team']
        
        try:
            # 獲取車手歷史序列數據
            sequence_data = self.get_driver_history_sequence(driver_name, circuit_name, year, df_processed=df_processed)
            if sequence_data is None:
                print(f"無法獲取 {driver_name} 的歷史數據")
                return None
            
            # 標準化序列數據
            sequence_reshaped = sequence_data.reshape(-1, sequence_data.shape[-1])
            sequence_scaled = self.scalers['lstm_sequence'].transform(sequence_reshaped)
            sequence_scaled = sequence_scaled.reshape(sequence_data.shape)
            
            # LSTM預測
            skills = self.lstm_skill_model.predict(sequence_scaled, verbose=0)[0]
            driver_skill = float(skills[0] * 100)
            team_skill = float(skills[1] * 100)
            
            combined_skill = (driver_skill + team_skill) / 2
            circuit_factor = self._get_circuit_factor(driver_name, circuit_name)
            adjusted_skill = combined_skill * circuit_factor
            
            return {
                'driver': driver_name,
                'team': current_team,
                'driver_skill': driver_skill,
                'team_skill': team_skill,
                'combined_skill': combined_skill,
                'adjusted_skill': adjusted_skill,
                'circuit_factor': circuit_factor,
                'confidence': self._calculate_confidence(driver_name, current_team),
                'prediction_method': 'LSTM'
            }
            
        except Exception as e:
            print(f"預測 {driver_name} 時發生錯誤: {e}")
            return None

    
    def _get_circuit_factor(self, driver_name, circuit_name):
        """計算特定賽道的適應性係數"""
        
        driver_hash = hash(driver_name) % 100
        circuit_hash = hash(circuit_name) % 100
        
        base_factor = 0.985 + (driver_hash + circuit_hash) % 31 / 1000  
        
        all_grand_prix = [
            'Great Britain', 'Monaco', 'Indianapolis 500', 'Switzerland', 'Belgium', 'France', 'Italy', 'Germany', 'Spain',
            'Netherlands', 'Argentina', 'Pescara', 'Portugal', 'Morocco', 'United States', 'South Africa', 'Mexico', 'Austria',
            'Canada', 'Brazil', 'Jarama', 'Sweden', 'Monza', 'Nürburgring', 'Watkins Glen', 'Kyalami', 'Long Beach', 'Japan',
            'Las Vegas', 'Dallas', 'Hungary', 'Australia', 'Phoenix', 'San Marino', 'Detroit', 'Europe', 'Luxembourg', 'Malaysia',
            'Bahrain', 'China', 'Singapore', 'Abu Dhabi', 'Korea', 'India', 'Russia', 'Azerbaijan', 'Qatar', 'Miami',
            'Saudi Arabia', 'Turkey', 'Emilia Romagna', 'Styria', 'Tuscan', '70th Anniversary', 'Eifel', 'Sakhir'
        ]
        
        circuit_adjustments = {gp: 0 for gp in all_grand_prix}
        circuit_adjustments.update({
            # 技術型賽道（有利於技術好的車手）
            'Monaco': 0.02,          # 從 0.01 增加到 0.02
            'Singapore': 0.015,      # 新增：街道賽，技術要求高
            'Hungary': 0.012,        # 新增：狹窄賽道，超車困難
            'Azerbaijan': 0.008,     # 新增：街道賽特性
            
            # 高速賽道（有利於膽量大的車手）
            'Italy': -0.015,         # 從 -0.007 增加到 -0.015
            'Belgium': 0.015,        # 從 0.008 增加到 0.015
            'Austria': 0.010,        # 新增：高海拔，引擎功率重要
            'Brazil': 0.018,         # 從 0.009 增加到 0.018
            
            # 傳統賽道
            'Great Britain': 0.012,  # 從 0.005 增加到 0.012
            'Germany': 0.010,        # 從 0.004 增加到 0.010
            'Netherlands': -0.008,   # 從 -0.004 增加到 -0.008
            
            # 新增其他特色賽道
            'Miami': -0.010,         # 新賽道，經驗不足影響大
            'Saudi Arabia': 0.015,   # 高速街道賽
            'Abu Dhabi': 0.008,      # 薄暮比賽，視線要求高
            'Canada': 0.012,         # 需要精準的煞車點
            'Japan': 0.014,          # 技術賽道，天氣變化大
            'Mexico': -0.006,        # 高海拔影響引擎表現
            'Australia': 0.006,      # 賽季首場，狀態調整重要
        })
        
        adjustment = circuit_adjustments.get(circuit_name, 0)
        
        # 方案3: 擴大最終係數範圍（從6%增加到12%）
        final_factor = base_factor + adjustment
        return max(0.88, min(1.12, final_factor)) 



    def _calculate_confidence(self, driver_name, team_name):
        """計算信心度"""
        if driver_name not in self.driver_team_history:
            return 0.5
            
        total_races = self.active_drivers.get(driver_name, {}).get('total_races', 0)
        team_races = self.driver_team_history[driver_name].get(team_name, 0)
        
        data_confidence = min(total_races / 50, 1.0)
        team_confidence = min(team_races / 20, 1.0)
        
        return (data_confidence + team_confidence) / 2

    def predict_race_ranking(self, circuit_name, year, selected_drivers, top_n=10):
        """預測比賽排名（基於綜合實力）"""
        print(f"預測 {circuit_name} {year} 年比賽，選定 {len(selected_drivers)} 位車手")
        
        # 先統一前處理一次
        df_processed = self.create_skill_features(self.df)

        results = []
        for driver_name in selected_drivers:
            result = self.predict_driver_performance(driver_name, circuit_name, year, df_processed=df_processed)
            if result:
                results.append(result)
        
        if not results:
            return []
        
        # 為每個車手添加比賽日隨機性
        for result in results:
            race_day_factor = np.random.normal(1.0, 0.1)
            result['race_day_skill'] = result['adjusted_skill'] * race_day_factor
            result['race_day_skill'] = max(0, min(100, result['race_day_skill']))
        
        # 根據比賽日綜合實力排序
        results.sort(key=lambda x: x['race_day_skill'], reverse=True)
        
        # 分配最終排名並限制輸出數量
        final_results = []
        for i, result in enumerate(results[:top_n]):
            result['final_position'] = i + 1
            final_results.append(result)
            
        return final_results



def create_gradio_interface(prediction_system):
    """創建Gradio界面"""
    
    def format_prediction_results(circuit, year, selected_drivers, top_n):
        if not selected_drivers:
            return "請選擇至少一位車手進行預測"
            
        try:
            results = prediction_system.predict_race_ranking(
                circuit, int(year), selected_drivers, int(top_n))
            
            if not results:
                return "預測失敗，請檢查選擇的車手"
            
            # 格式化輸出
            output = f"## 🏁 {circuit} {year}年 比賽預測結果 (前{len(results)}名)\n\n"
            
            output += "| 排名 | 車手 | 當前車隊 | 車手實力 | 車隊實力 | 綜合實力 | 賽道適應 | 比賽日表現 |\n"
            output += "|------|------|----------|----------|----------|----------|----------|------------|\n"
            
            for result in results:
                confidence_pct = result['confidence'] * 100
                circuit_factor_pct = (result['circuit_factor'] - 1) * 100
                
                output += f"| {result['final_position']} | {result['driver']} | {result['team']} | "
                output += f"{result['driver_skill']:.1f} | {result['team_skill']:.1f} | "
                output += f"{result['combined_skill']:.1f} | {circuit_factor_pct:+.1f}% | "
                output += f"{result['race_day_skill']:.1f} \n"
            
            # 添加預測邏輯說明
            output += f"\n\n### 📊 {year}年 預測邏輯說明\n"
            output += f"""
**{year}年活躍車手篩選標準:**
- 在 {year-5} - {year+2} 年期間有比賽記錄的車手
- 根據時間距離計算活躍度分數，優先顯示該年份最相關的車手
- 自動匹配車手在 {year} 年的車隊歸屬

**排名計算方式:**
1. **基礎實力**: 車手實力與車隊實力的平均值
2. **賽道適應**: 根據車手特性和賽道特點調整實力
3. **比賽日隨機性**: 模擬比賽中的意外情況和表現波動
4. **最終排名**: 按比賽日綜合表現排序

**實力計算:**
- **車手實力**: 基於歷史比賽成績，使用指數移動平均突出近期表現
- **車隊實力**: 基於車隊歷年平均表現，反映技術水平和資源
- **綜合實力**: 車手實力與車隊實力的平均值

**特殊調整:**
- **賽道適應**: 不同車手在不同賽道的表現差異（±20%調整範圍）
- **比賽日波動**: 模擬比賽當天的各種不確定因素（±5%隨機波動）

            """
            
            return output
            
        except Exception as e:
            return f"預測過程中發生錯誤: {str(e)}"

    def update_drivers_for_year(year):
        """根據年份更新活躍車手列表"""
        try:
            active_driver_list = prediction_system.update_active_drivers_for_year(year)
            # 默認選擇前10名
            default_selection = active_driver_list[:10] if len(active_driver_list) >= 10 else active_driver_list
            return gr.Dropdown(choices=active_driver_list, value=default_selection)
        except Exception as e:
            print(f"更新車手列表時發生錯誤: {e}")
            return gr.Dropdown(choices=[], value=[])

    def get_active_drivers_info_for_year(year):
        """獲取指定年份的活躍車手信息"""
        try:
            year_int = int(year)
            active_drivers = prediction_system.get_active_drivers_for_year(year_int)
            
            info_text = f"### 👥 {year_int}年 活躍車手一覽 ({len(active_drivers)}位)\n\n"
            info_text += "| 車手 | 車隊 | 總比賽數 | 活躍期間 | 活躍度 | 狀態 |\n"
            info_text += "|------|------|----------|----------|--------|------|\n"
            
            # 按活躍度分數排序
            sorted_drivers = sorted(active_drivers.items(), 
                                  key=lambda x: x[1]['activity_score'], reverse=True)
            
            for driver, info in sorted_drivers[:20]:  # 顯示前20名
                status = "正當年" if info['first_active_year'] <= year_int <= info['last_active_year'] else \
                        f"退役{info['years_since_last_race']}年" if info['years_since_last_race'] > 0 else \
                        f"{abs(info['years_since_last_race'])}年後出道"
                
                active_period = f"{info['first_active_year']}-{info['last_active_year']}"
                
                info_text += f"| {driver} | {info['current_team']} | {info['total_races']} | "
                info_text += f"{active_period} | {info['activity_score']:.2f} | {status} |\n"
            
            if len(active_drivers) > 20:
                info_text += f"\n*還有 {len(active_drivers) - 20} 位活躍車手...*"
                
            return info_text
            
        except Exception as e:
            return f"獲取車手信息時發生錯誤: {str(e)}"
    
    # 獲取初始活躍車手列表
    initial_year = 2024
    try:
        active_driver_list = prediction_system.update_active_drivers_for_year(initial_year)
    except:
        active_driver_list = list(prediction_system.active_drivers.keys()) if hasattr(prediction_system, 'active_drivers') else []
        active_driver_list.sort()
    
    with gr.Blocks(title="F1 比賽預測系統", theme=gr.themes.Soft()) as interface:
        gr.Markdown("# 🏎️ F1 比賽預測系統")
        gr.Markdown("""
        ### 🎯 系統特色
        - **綜合實力排名**: 基於車手實力、車隊實力和賽道適應性進行排名
        - **智能車隊匹配**: 自動為車手匹配當前車隊，無需手動選擇
        - **活躍車手篩選**: 只顯示近期活躍的車手，確保預測相關性  
        - **比賽日模擬**: 添加隨機因素模擬真實比賽的不確定性
        - **透明預測邏輯**: 詳細說明實力計算和排名預測方法
        """)
        
        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### ⚙️ 比賽設定")
                
                circuit_dropdown = gr.Dropdown(
                    choices=prediction_system.circuit_names if hasattr(prediction_system, 'circuit_names') else [],
                    label="🏁 選擇賽道",
                    value=prediction_system.circuit_names[0] if hasattr(prediction_system, 'circuit_names') and prediction_system.circuit_names else None
                )
                
                year_slider = gr.Slider(
                    minimum=1960,
                    maximum=2030,
                    step=1,
                    value=initial_year,
                    label="📅 比賽年份"
                )
                
                drivers_dropdown = gr.Dropdown(
                    choices=active_driver_list,
                    label="🏎️ 選擇參賽車手",
                    multiselect=True,
                    
                )
                
                select_all_btn = gr.Button("全選/取消全選")

                # 全選/取消全選 callback
                def toggle_select_all_drivers(selected_drivers, year):
                    try:
                        # 獲取當前年份的活躍車手列表
                        current_active_drivers = prediction_system.update_active_drivers_for_year(year)
                        
                        # 檢查是否已經全選
                        if selected_drivers and set(selected_drivers) == set(current_active_drivers):
                            # 如果已經全選→改成全取消
                            return []
                        else:
                            # 否則就全選
                            return current_active_drivers
                    except Exception as e:
                        print(f"全選操作發生錯誤: {e}")
                        return selected_drivers or []

                select_all_btn.click(
                    fn=toggle_select_all_drivers,
                    inputs=[drivers_dropdown, year_slider],
                    outputs=drivers_dropdown
                )
                
                top_n_slider = gr.Slider(
                    minimum=3,
                    maximum=20,
                    step=1,
                    value=10,
                    label="🏆 顯示排名數量",
                    info="選擇要顯示的前N名結果"
                )
                
                predict_button = gr.Button("🚀 開始預測", variant="primary", size="lg")
                
                # 系統統計
                gr.Markdown(f"""
                **📈 系統數據:**
                - 可選賽道: {len(prediction_system.circuit_names) if hasattr(prediction_system, 'circuit_names') else 0} 個
                - 訓練數據: {len(prediction_system.training_data) if hasattr(prediction_system, 'training_data') and prediction_system.training_data is not None else 0} 筆記錄
                """)
                
            with gr.Column(scale=2):
                gr.Markdown("### 🏆 預測結果")
                results_output = gr.Markdown(
                    value="選擇車手和比賽參數，然後點擊「開始預測」查看結果",
                    elem_classes=["prediction-output"]
                )
        
        # 事件處理
        predict_button.click(
            fn=format_prediction_results,
            inputs=[circuit_dropdown, year_slider, drivers_dropdown, top_n_slider],
            outputs=[results_output]
        )
        
        # 年份變更時更新車手列表
        year_slider.change(
            fn=update_drivers_for_year,
            inputs=[year_slider],
            outputs=[drivers_dropdown]
        )
        
        # 活躍車手信息展示
        with gr.Row():
            active_drivers_info = gr.Markdown(
                value=get_active_drivers_info_for_year(initial_year)
            )
        
        # 年份變更時更新活躍車手信息
        year_slider.change(
            fn=get_active_drivers_info_for_year,
            inputs=[year_slider],
            outputs=[active_drivers_info]
        )
    
    return interface

In [ ]:
def main():
    print("載入F1數據...")
    try:
        df = pd.read_csv('./data/f1_merged_all.csv')
        print(f"成功載入數據: {df.shape}")
    except FileNotFoundError:
        print("未找到 f1_merged_all.csv，創建示例數據...")
        np.random.seed(42)
        drivers = ['Hamilton', 'Verstappen', 'Leclerc', 'Russell', 'Sainz', 'Norris', 'Piastri', 'Alonso']
        teams = ['Mercedes', 'Red Bull', 'Ferrari', 'McLaren', 'Aston Martin']
        circuits = ['Monaco', 'Silverstone', 'Monza', 'Spa', 'Suzuka']
        years = [2020, 2021, 2022, 2023, 2024]
        data = []
        for year in years:
            for circuit in circuits:
                for i, driver in enumerate(drivers):
                    team = teams[i % len(teams)]
                    position = np.random.randint(1, 21)
                    data.append({
                        'Driver': driver,
                        'Team': team,
                        'Grand Prix': circuit,
                        'year': year,
                        'Position': position,
                        'Fastest Lap Time': f"1:{np.random.randint(15, 45)}.{np.random.randint(100, 999)}"
                    })
        df = pd.DataFrame(data)
    prediction_system = F1PredictionSystem()
    prediction_system.train_models(df)
    interface = create_gradio_interface(prediction_system)
    print("啟動Gradio界面...")
    interface.launch()

if __name__ == '__main__':
    main()


載入F1數據...
成功載入數據: (2284, 12)
開始訓練LSTM模型...
準備序列數據，序列長度: 5
準備數據中...
存儲車手完整數據...
存儲了 399 位車手的完整數據
識別活躍車手...
獲取 2024 年的活躍車手...
為 2024 年選定 34 位活躍車手
初始識別出 34 位活躍車手
計算實力特徵...
使用位置欄位: Pos

=== 標準化前後對照 ===
標準化前前5名:
  Max  Verstappen: 89.88
  Juan Manuel  Fangio: 89.85
  Alain  Prost: 86.18
  Jackie  Stewart: 85.57
  Lewis  Hamilton: 84.97
標準化後前5名:
  Max  Verstappen: 100.00
  Juan Manuel  Fangio: 99.97
  Alain  Prost: 95.85
  Jackie  Stewart: 95.16
  Lewis  Hamilton: 94.49
準備了 1129 個序列
訓練LSTM實力預測模型...
Epoch 1/100
29/29 [==============================] - 4s 19ms/step - loss: 0.0215 - mae: 0.1560 - val_loss: 0.0221 - val_mae: 0.1736 - lr: 0.0010
Epoch 2/100
29/29 [==============================] - 0s 8ms/step - loss: 0.0098 - mae: 0.1051 - val_loss: 0.0220 - val_mae: 0.1738 - lr: 0.0010
Epoch 3/100
29/29 [==============================] - 0s 8ms/step - loss: 0.0082 - mae: 0.0966 - val_loss: 0.0210 - val_mae: 0.1693 - lr: 0.0010
Epoch 4/100
29/29 [==============================] - 0s 8ms/step - lo

獲取 2024 年的活躍車手...
為 2024 年選定 34 位活躍車手
預測 70th Anniversary 2024 年比賽，選定 34 位車手
計算實力特徵...
使用位置欄位: Pos

=== 標準化前後對照 ===
標準化前前5名:
  Max  Verstappen : 89.88
  Juan Manuel  Fangio : 89.85
  Alain  Prost : 86.18
  Jackie  Stewart : 85.57
  Lewis  Hamilton : 84.97
標準化後前5名:
  Max  Verstappen : 100.00
  Juan Manuel  Fangio : 99.97
  Alain  Prost : 95.85
  Jackie  Stewart : 95.16
  Lewis  Hamilton : 94.49
獲取 2023 年的活躍車手...
為 2023 年選定 38 位活躍車手
獲取 2022 年的活躍車手...
為 2022 年選定 42 位活躍車手
獲取 2013 年的活躍車手...
為 2013 年選定 59 位活躍車手
獲取 2013 年的活躍車手...
為 2013 年選定 59 位活躍車手
獲取 2002 年的活躍車手...
為 2002 年選定 52 位活躍車手
獲取 2002 年的活躍車手...
為 2002 年選定 52 位活躍車手
獲取 1996 年的活躍車手...獲取 1996 年的活躍車手...
為 1996 年選定 58 位活躍車手

為 1996 年選定 58 位活躍車手
獲取 1995 年的活躍車手...
為 1995 年選定 57 位活躍車手
獲取 1995 年的活躍車手...
為 1995 年選定 57 位活躍車手
預測 70th Anniversary 1995 年比賽，選定 10 位車手
計算實力特徵...
使用位置欄位: Pos

=== 標準化前後對照 ===
標準化前前5名:
  Max  Verstappen : 89.88
  Juan Manuel  Fangio : 89.85
  Alain  Prost : 86.18
  Jackie  Stewart : 85.57
  Lewis  Hamilton : 84.97
標準化後前